In [0]:
# Retail Sales Lakehouse - Incremental Load


from pyspark.sql.functions import *
from delta.tables import DeltaTable

catalog = "workspace"
schema = "default"

print("Incremental Load setup completed")

In [0]:
customers_before = spark.table("workspace.default.silver_customers")

print("Current Silver Customers:", customers_before.count())

display(
    customers_before.orderBy("customer_id")
)

In [0]:
from datetime import date

incremental_data = [
    # Existing customer - UPDATE scenario
    (1001, "Arun Kumar", "arun.kumar@email.com", "Cambridge", "UK", date(2025, 1, 10)),

    # New customer - INSERT scenario
    (1009, "Harry Clark", "harry.clark@email.com", "Oxford", "UK", date(2026, 8, 20))
]

incremental_columns = [
    "customer_id",
    "customer_name",
    "email",
    "city",
    "country",
    "signup_date"
]

incremental_customers_df = spark.createDataFrame(
    incremental_data,
    incremental_columns
)

display(incremental_customers_df)

In [0]:
from delta.tables import DeltaTable

target_table = DeltaTable.forName(
    spark,
    "workspace.default.silver_customers"
)

target_table.alias("target") \
    .merge(
        incremental_customers_df.alias("source"),
        "target.customer_id = source.customer_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

print("Delta MERGE completed successfully")

In [0]:
customers_after = spark.table("workspace.default.silver_customers")

print("Customers after MERGE:", customers_after.count())

display(
    customers_after
    .filter(col("customer_id").isin(1001, 1009))
    .orderBy("customer_id")
)

In [0]:
from datetime import datetime

watermark_data = [
    (
        "customers",
        datetime(2026, 8, 20, 0, 0, 0)
    )
]

watermark_columns = [
    "source_name",
    "last_processed_timestamp"
]

watermark_df = spark.createDataFrame(
    watermark_data,
    watermark_columns
)

watermark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.etl_watermark")

display(
    spark.table("workspace.default.etl_watermark")
)

In [0]:
watermark_table = spark.table("workspace.default.etl_watermark")

last_processed_ts = (
    watermark_table
    .filter(col("source_name") == "customers")
    .select("last_processed_timestamp")
    .collect()[0][0]
)

print("Last processed timestamp:", last_processed_ts)

In [0]:
from pyspark.sql.functions import current_timestamp

incoming_customers = (
    incremental_customers_df
    .withColumn("ingestion_timestamp", current_timestamp())
)

display(incoming_customers)

In [0]:
new_incremental_records = (
    incoming_customers
    .filter(col("ingestion_timestamp") > last_processed_ts)
)

print("New records after watermark filter:", new_incremental_records.count())

display(new_incremental_records)

In [0]:
from pyspark.sql.functions import max as spark_max

new_watermark = (
    new_incremental_records
    .agg(spark_max("ingestion_timestamp").alias("max_ts"))
    .collect()[0]["max_ts"]
)

updated_watermark_df = spark.createDataFrame(
    [("customers", new_watermark)],
    ["source_name", "last_processed_timestamp"]
)

updated_watermark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.etl_watermark")

display(spark.table("workspace.default.etl_watermark"))

In [0]:
latest_watermark = (
    spark.table("workspace.default.etl_watermark")
    .filter(col("source_name") == "customers")
    .select("last_processed_timestamp")
    .collect()[0][0]
)

reprocessed_records = (
    incoming_customers
    .filter(col("ingestion_timestamp") > latest_watermark)
)

print("Records eligible on re-run:", reprocessed_records.count())

In [0]:
from pyspark.sql.functions import lit, current_timestamp

scd_customers = (
    spark.table("workspace.default.silver_customers")
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
    .withColumn("is_current", lit(True))
)

scd_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.dim_customers_scd2")

display(
    spark.table("workspace.default.dim_customers_scd2")
    .orderBy("customer_id")
)

In [0]:
display(
    spark.table("workspace.default.dim_customers_scd2")
    .filter(col("customer_id") == 1001)
    .select(
        "customer_id",
        "customer_name",
        "city",
        "effective_from",
        "effective_to",
        "is_current"
    )
)

In [0]:
from datetime import date

scd_change_data = [
    (
        1001,
        "Arun Kumar",
        "arun.kumar@email.com",
        "Edinburgh",
        "UK",
        date(2025, 1, 10)
    )
]

scd_change_columns = [
    "customer_id",
    "customer_name",
    "email",
    "city",
    "country",
    "signup_date"
]

scd_change_df = spark.createDataFrame(
    scd_change_data,
    scd_change_columns
)

display(scd_change_df)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp

scd_table = DeltaTable.forName(
    spark,
    "workspace.default.dim_customers_scd2"
)

scd_table.update(
    condition="""
        customer_id = 1001
        AND is_current = true
    """,
    set={
        "effective_to": "current_timestamp()",
        "is_current": "false"
    }
)

print("Old customer version closed successfully")

In [0]:
from pyspark.sql.functions import lit, current_timestamp

new_customer_version = (
    scd_change_df
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
    .withColumn("is_current", lit(True))
)

new_customer_version.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.default.dim_customers_scd2")

print("New customer version inserted successfully")

In [0]:
customer_history = (
    spark.table("workspace.default.dim_customers_scd2")
    .filter(col("customer_id") == 1001)
    .select(
        "customer_id",
        "customer_name",
        "city",
        "effective_from",
        "effective_to",
        "is_current"
    )
    .orderBy("effective_from")
)

display(customer_history)